<div style="border-left:4px solid #34d399;padding:2px 0 2px 16px;margin:6px 0 18px;"><div style="font:800 27px/1.15 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;letter-spacing:-0.02em;">NL2SQL <span style="font-weight:500;color:#34d399;">Understanding</span></div><div style="font:400 15px/1.55 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#71717a;margin-top:5px;">How an English question becomes a schema, a symbol and an exact value.</div></div>

[Setup](https://www.kaggle.com/code/kirazul/nl2sql-1-setup) &nbsp;|&nbsp; **Understanding** &nbsp;|&nbsp; [Architectures](https://www.kaggle.com/code/kirazul/nl2sql-3-architectures) &nbsp;|&nbsp; [Run All](https://www.kaggle.com/code/kirazul/nl2sql-4-run-all)

## 1. Setup

The code is cloned from GitHub. The database, the index and the two models are
read from [notebook 1](https://www.kaggle.com/code/kirazul/nl2sql-1-setup)'s saved output, where they already are. Nothing
is downloaded or rebuilt here.

Before running: **Add Input > Notebook Output > NL2SQL 1 Setup**, add the secrets
listed below under **Add-ons > Secrets**, and enable Internet.

In [ ]:
%%capture --no-stderr
!pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu \
    "llama-cpp-python>=0.3" "gliner2>=1.3" "langgraph>=1.0" "langsmith>=0.10" \
    "fastapi>=0.115" "uvicorn[standard]>=0.34" "pydantic-settings>=2.6" \
    "sqlglot>=25.0" "rapidfuzz>=3.10" "pyyaml>=6.0" "httpx>=0.27" "python-dotenv>=1.0"

In [ ]:
import os, re, sys, json, time, shutil, subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
WORK      = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
INPUTS    = Path("/kaggle/input")
REPO      = "https://github.com/Kirazul/NL2SQL-demo.git"

SECRETS = {
    "GITHUB_TOKEN":       "clone the code (the repository is private)",
    "GROQ_API_KEY":       "the cloud model that writes the SQL",
    "OPENROUTER_API_KEY": "fallback when Groq rate-limits",
    "LANGSMITH_API_KEY":  "tracing backend",
    "PUBLISH_TOKEN":      "announce this session to the web interface",
}
REQUIRED = ()


WHY = {}          # label -> why it could not be read, when it could not


def secret(label, default=""):
    """One secret, by label. Kaggle grants access per notebook, not per account.

    The reason a lookup failed is kept rather than swallowed: "not attached to
    this notebook" and "the backend refused" both end as an empty string, and
    without the reason the two are indistinguishable from the output.
    """
    if ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(label)
            if value:
                return value
            WHY[label] = "Kaggle returned an empty value"
        except Exception as error:
            WHY[label] = f"{type(error).__name__}: {str(error)[:110]}"
    return os.environ.get(label, default)


def load_secrets(project=None):
    """Read every label into the environment and print what was found.

    An empty secret is removed rather than set blank, so the package falls back to
    its own default instead of an empty string.
    """
    local = {}
    if not ON_KAGGLE and project and (project / ".env").exists():
        for line in (project / ".env").read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                label, _, value = line.partition("=")
                local[label.strip()] = value.strip().strip("\"'")

    for label in SECRETS:
        value = secret(label) or local.get(label, "")
        if value:
            os.environ[label] = value
        else:
            os.environ.pop(label, None)

    for label, purpose in SECRETS.items():
        if os.environ.get(label):
            state = "ok"
        elif label in REQUIRED:
            state = "REQUIRED"
        else:
            state = "-"
        print(f"  {label:<20}{state:<10}{purpose}")

    if WHY:
        print("\n  why a secret could not be read")
        for label, reason in WHY.items():
            print(f"    {label:<20}{reason}")

    absent = [l for l in REQUIRED if not os.environ.get(l)]
    if absent:
        where = "Add-ons > Secrets, in this notebook" if ON_KAGGLE else ".env"
        print(f"\n  Missing: {', '.join(absent)}. Set it in {where} and run this cell again.")
    return not absent


def get_code():
    """Clone the repository into a writable directory and put it on the path.

    Kaggle mounts every input read-only and notebook 1 writes a database next to
    the package, so the code never runs from where it is mounted.
    """
    if (Path.cwd() / "src/hybridsql").exists():
        return Path.cwd()

    target = WORK / "nl2sql"
    if (target / "src/hybridsql").exists():
        return target

    token = secret("GITHUB_TOKEN")
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    done = subprocess.run(["git", "clone", "--depth", "1", "--quiet", url, str(target)],
                          capture_output=True, text=True)
    if done.returncode:
        detail = done.stderr.replace(token, "***") if token else done.stderr
        raise SystemExit(
            "Could not clone the repository.\n\n"
            "  It is private, so this notebook needs a GITHUB_TOKEN secret:\n"
            "  Add-ons > Secrets > attach GITHUB_TOKEN, then run this cell again.\n\n"
            "  Kaggle grants a secret one notebook at a time. Attaching it in\n"
            "  another notebook does not attach it here.\n\n" + detail
        )
    return target
ARTEFACTS = {
    "database":    ("data/warehouse/eicu.db",                   None),
    "value index": ("data/warehouse/value_index.db",            None),
    "GLiNER2":     ("models/gliner2-base-v1",                   "model.safetensors"),
    "Qwen3-1.7B":  ("models/qwen3-1.7b/Qwen3-1.7B-Q4_K_M.gguf", None),
}
DEPTHS = ("", "*/", "*/*/", "*/*/*/", "*/*/*/*/", "*/*/*/*/*/")


def whole(path, probe=None):
    """Present and finished. A model directory with no weights in it is neither."""
    return (path / probe).exists() if probe else path.exists()


def find_input(relative, probe=None):
    """The first attached input carrying `relative`, at whatever depth it sits."""
    if not INPUTS.exists():
        return None
    for prefix in DEPTHS:
        for hit in sorted(INPUTS.glob(prefix + relative)):
            if whole(hit, probe):
                return hit
    return None


def attached_inputs():
    """The inputs actually attached, named by what they carry rather than by the
    directory level Kaggle happens to mount them under."""
    if not INPUTS.exists():
        return []
    markers = ("src", "data", "models", "nl2sql")
    return [p.relative_to(INPUTS).as_posix()
            for pattern in ("*", "*/*", "*/*/*")
            for p in sorted(INPUTS.glob(pattern))
            if p.is_dir() and any((p / m).exists() for m in markers)]


def locate(project):
    """Every artefact, in the working copy or in an attached input."""
    found, missing = {}, []
    for label, (relative, probe) in ARTEFACTS.items():
        if label == "value index":
            continue                       # always beside the database, see below
        local = project / relative
        path = local if whole(local, probe) else find_input(relative, probe)
        (found.__setitem__(label, path) if path else missing.append(label))

    # The package derives the index path from the database path, so the two must
    # be in the same directory. Looking for it anywhere else would resolve here
    # and fail there.
    if "database" in found:
        index = found["database"].with_name("value_index.db")
        found["value index"] = index if index.exists() else missing.append("value index")
    else:
        missing.append("value index")
    return found, [m for m in missing if m]


def configure(found):
    os.environ["DB_PATH"]             = str(found["database"])
    os.environ["GLINER_MODEL"]        = str(found["GLiNER2"])
    os.environ["LOCAL_LLM_GGUF_PATH"] = str(found["Qwen3-1.7B"])
    os.environ["LOCAL_LLM_THREADS"]   = str(max(2, os.cpu_count() or 4))
    os.environ["LOCAL_LLM_BACKEND"]   = "llamacpp"
    os.environ["PRIVACY_MODE"]        = "demo"
    os.environ["LANGSMITH_PROJECT"]   = "nl2sql"
    os.environ["LANGSMITH_TRACING"]   = "1" if os.environ.get("LANGSMITH_API_KEY") else "0"


def size_mb(path):
    if path.is_dir():
        return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e6
    return path.stat().st_size / 1e6 if path.exists() else 0.0


def show(found):
    for label, (relative, _) in ARTEFACTS.items():
        path = found.get(label)
        if path is None:
            print(f"  {label:<14}{'missing':>10}")
            continue
        root = path.parents[len(Path(relative).parts) - 1]
        if WORK in path.parents:
            where = "built here"
        elif INPUTS.exists() and (INPUTS in root.parents or root == INPUTS):
            where = root.relative_to(INPUTS).as_posix()
        else:
            where = str(root)
        print(f"  {label:<14}{size_mb(path):>9.0f} MB   {where}")
print("code")
PROJECT = get_code()
sys.path.insert(0, str(PROJECT / "src"))
os.chdir(PROJECT)
print(f"  {PROJECT}")

print("\nsecrets")
load_secrets(PROJECT)

FOUND, MISSING = locate(PROJECT)
if MISSING:
    raise SystemExit(
        "Notebook 1's output is not attached, and nothing is built in this notebook.\n"
        f"  missing:  {', '.join(MISSING)}\n"
        f"  attached: {attached_inputs() or 'nothing'}\n\n"
        "  Add Input > Notebook Output > NL2SQL 1 Setup\n"
        "  https://www.kaggle.com/code/kirazul/nl2sql-1-setup"
    )

configure(FOUND)
print("\nartefacts")
show(FOUND)

---

## 2. The problem

Someone types, in English:

> *How many patients over 65 received aspirin?*

To answer it we need SQL. Writing good SQL over 31 unfamiliar tables is something
large cloud models do well and small local models do badly. But sending the
question to a cloud model sends what we are protecting, because the question *is
about the data*: it names a drug, an age, a ward.

So the question is taken apart **here**, on this machine, before anything is sent.
Four steps, four different jobs:

| Step | The question it answers | Tool | Section |
|---|---|---|---|
| 1 | which **words** matter? | GLiNER2, a small local model | 4 |
| 2 | is a word a **stored value**? | the value index | 5 |
| 3 | is a word a **column name**? | the column catalogue | 6 |
| 4 | so which is it? | the arbitration | 7 |
| 5 | replace every value with a symbol | the masking stage | 8 |
| 6 | may what is left **leave**? | the egress gate | 9 |

At the end, the question has become four things and none of them is a secret: the
tables, the column names, the values (kept here, replaced by symbols), and the
sentence with holes in it.

---

## 3. The database

31 tables of intensive-care records. Almost every one hangs off a single column:
`patientunitstayid` identifies one patient's stay in one ICU and appears in 28 of
them. That is what makes *which patients on aspirin had a high creatinine*
answerable — `medication` and `lab` both carry it, so they join.

In [ ]:
from hybridsql.db import schema as sch

for key, value in sch.summary().items():
    print(f"  {key:<16}{value:,}")

print("\n" + sch.ddl({"medication"}, with_row_counts=True)[:430])

The DDL above is **the entire disclosure** of the protected architecture: table
names, column names, types, row counts, foreign keys. Not one value from any row.

---

## 4. Step 1 — find the words that matter

**GLiNER2** is a 208 M-parameter model that reads a sentence and returns the spans
that carry meaning. Two properties are why it is here instead of an LLM:

- it runs **inside this process, on the CPU**, in about 100 ms — so the question
  never leaves the machine to be understood;
- it is **zero-shot**: the entity types are written in plain English at call time.
  Changing domain means editing a list of strings, not retraining anything.

It finds *where* something interesting is. It does not know this database, and it
returns `drug`, not `medication.drugname`. The next two steps do that.

Two limits, both measured rather than assumed. **The span is a guess**: it returns
`hemoglobin lab test` where only `hemoglobin` is stored, so shorter spans are tried
against the index rather than trusting the boundaries. And **it does not see numbers
as data**: `hospital id 56` came back as one entity and `56` went out in clear text,
so numbers are found by a separate regular expression, which cannot miss one.

In [ ]:
from hybridsql.providers import extractor

print(f"  model  {extractor.state()['model']}   loaded in {extractor.state()['load_ms']} ms")
print(f"  types  {', '.join(extractor.ENTITY_TYPES[:4])}, +{len(extractor.ENTITY_TYPES) - 4} more\n")

for question in ["How many patients over 65 received aspirin?", "Did Mr. Bensalah get his insulin?"]:
    print(f"  {question}")
    for e in extractor.extract(question):
        print(f"      {e.text:<24}{e.type:<12}{e.score:.2f}")
    print()

---

## 5. Step 2 — is the word a stored value?

The analyst writes *aspirin*. The database stores `ASPIRIN EC 81 MG PO TBEC`. A
query written `WHERE drugname = 'aspirin'` returns **nothing**. An index built once
by notebook 1 closes that gap locally — and it is the same step that tells us which
exact string must never leave.

### Why not index every value

Because then the cost grows with the number of rows, and a privacy design that
stops working on a big database is not a design. Every text column is measured once
and sorted into three tiers.

| Tier | The column looks like | Stored |
|---|---|---|
| **A** | a bounded vocabulary — 1 402 drug names, 12 ward types | every distinct value |
| **B** | thousands of distinct strings | **nothing** — searched on demand |
| **C** | identifiers, timestamps, free text | **nothing** — never searched |

**That is the whole scalability argument.** Adding ten million rows adds nothing to
the index. If a column outgrows the limit it moves to tier B and stores *less*.

In [ ]:
from hybridsql.db import value_index

s = value_index.stats()
total = sum(s["tiers"].values())
print(f"  text columns examined  {total}")
print(f"  tier A / B / C         {s['tiers']['A']} indexed, {s['tiers'].get('B', 0)} on demand, "
      f"{s['tiers'].get('C', 0)} excluded")
print(f"  values stored          {s['values_indexed']:,}")
print(f"  index size             {s['size_mb']} MB")
print(f"  ceiling, any row count {total * value_index.VOCABULARY_LIMIT * 83 / 1e6:.0f} MB\n")

for mention in ["aspirin", "asspirin"]:
    hit = value_index.search(mention, limit=1)[0]
    print(f"  {mention:<12}-> {hit.ref:<26}{hit.value!r}  ({hit.score:.2f})")

### The scorer, and the bug that lived in it

Similarity here is **not symmetric**, and treating it as if it were was the worst
defect this pipeline has had.

The legitimate case is the analyst naming *part* of a longer value: "aspirin" for
`ASPIRIN EC 81 MG PO TBEC`. The reverse — a short stored value found inside a long
question — is not a match at all. Measured, before the fix:

| The question said | What the index answered | Score |
|---|---|---|
| `10 most frequently recorded laboratory tests` | `10` | **1.00** |
| `laboratory records` | `lab` | **1.00** |
| `male patients` | `Older adult: Provide adequate time for patients…` | **0.76** |

All three cleared the confidence threshold, so all three were masked and sent to
the cloud model as filters. The queries ran and answered a different question.

Two rules fixed it: when the value is **shorter** than the mention only whole-string
similarity counts, and a compound value (`renal|acute renal failure|…`) is compared
**segment by segment** instead of as one long string that shares a word with
everything.

In [ ]:
print(f"  {'the question says':<44}{'the index now answers':<40}score")
for mention in ["10 most frequently recorded laboratory tests", "laboratory records",
                "male patients", "aspirin"]:
    hits = value_index.search(mention, limit=1)
    answer = f"{hits[0].ref} = {hits[0].value}"[:38] if hits else "nothing, which is correct"
    print(f"  {mention:<44}{answer:<40}{hits[0].score:.2f}" if hits
          else f"  {mention:<44}{answer:<40}")

---

## 6. Step 3 — is the word a column name?

The index answers *"which value is this?"*. It always answers something: a fuzzy
search over 30 000 values never comes back empty. So we also need the other
question — *"which column is this?"* — or every question that names a column gets
turned into a filter on an invented value. That is exactly what happened:

> *"What are the 10 most common **diagnosis names**?"*
> → matched `pasthistory.pasthistoryvalue = 'clinical diagnosis'` at 0.75, masked it,
> asked the cloud to filter on it, and came back with **one row**.

Column names in a real database are glued-together words — `labname`, `routeadmin`,
`nursingchartcelltypecat`. Splitting them would need a dictionary of the domain we
are trying to stay independent of, so matching runs on **character trigrams**
instead: `administration routes` and `medication.routeadmin` share `rou`, `out`,
`ute`, `adm`, `dmi`. No dictionary, no model, 2 ms, and it works on any schema.

In [ ]:
from hybridsql.db import catalog

print(f"  {catalog.stats()['columns_catalogued']} columns catalogued\n")
print(f"  {'the question says':<34}the columns it could mean")
for mention in ["diagnosis names", "medication administration routes",
                "hospital region", "ethnicity", "aspirin", "sepsis"]:
    shown = "  ".join(f"{m.ref} ({m.score:.2f})" for m in catalog.link(mention, limit=2))
    print(f"  {mention:<34}{shown or 'nothing — so it is not a column'}")

`aspirin` and `sepsis` are real values, and they match **no column at all**. That
gap is what the decision below runs on.

---

## 7. Step 4 — decide

One rule:

> A word is a **value** only if it matches a value better than it matches a column.
> A word stored in the database **exactly as typed** is never overruled.

The second clause is not a detail: `creatinine`, `glucose` and `albumin` are lab
values in `lab.labname` **and** column names in `apacheapsvar`.

Numbers skip all of this and are always masked. "Over 65" is a threshold the
analyst chose, but "hospital id 56" identifies one hospital, and the sentence does
not reliably tell them apart. The model never needs a number's value to write
`WHERE hospitalid = :v1`, so masking every one costs nothing.

In [ ]:
from hybridsql.pipeline.understand import understand

for question in ["How many patients over 65 received aspirin?",
                 "What are the 10 most common diagnosis names?",
                 "How many patients in hospital id 56?"]:
    r = understand(question)
    print(f"  {question}")
    for x in r.resolutions:
        kept = f"   kept here: {x.value!r}" if x.value else ""
        print(f"      {x.mention:<22}{x.kind:<10}{str(x.column or '-'):<32}{kept}")
    print()

Three kinds come out. **value** exists in the database, so it is a secret and gets
masked. **concept** names a column — not a secret, the column name is in the DDL
anyway, and masking it would leave the model unable to write the query. **quantity**
is a number, masked.

---

## 8. Masking

Each value and each number becomes a symbol. The mapping stays in this process, and
the symbols are **renumbered on every request** — `:v1` here and `:v1` in the next
question are unrelated, so nobody watching the traffic can follow a value across two
questions.

In [ ]:
from hybridsql.pipeline.anonymize import anonymize

u = understand("How many patients over 65 received aspirin?")
a = anonymize(u)

print(f"  asked   {u.question}")
print(f"  sent    {a.masked_question}\n")
print("  kept here, and nowhere else")
for symbol, value in a.mapping.items():
    print(f"    {symbol:<6}{value!r:<34}{a.columns.get(symbol, '')}")

---

## 9. The egress gate

Masking is the design. **The gate is what makes it checkable.**

> One rule: exactly one module may open a socket, and every piece of text it sends
> is verified first.

Not "should be" — the gate raises an exception, so an unverified send does not
happen.

### Checking by origin, not by vocabulary

The first version read the whole prompt word by word. It worked, and it forced
hundreds of ordinary English words into a hand-written list so that **our own
sentences** could pass. Every new question added one more word, and each addition
widened the hole.

The prompt is now split by where each part came from, and each part is checked by
the rule that can actually prove *that kind* of text safe. Three of the four are
reconstructions: the gate recomputes what the part should contain and compares.

| Origin | Proven safe by | Can it hide a value? |
|---|---|---|
| `authored` | SHA-256 of a literal written in the source | no, it is a constant |
| `template` | SHA-256 of the wording, labels blanked out | no, only `t3`, `c7`, numbers vary |
| `schema` | regenerating the DDL from the database and comparing | no, the DDL holds no row |
| `glossary` | membership among the declared notes | no, a modified note is not a note |
| `question` | **word by word, fail-closed** | yes — the only untrusted part |

In [ ]:
from hybridsql.pipeline import generate as gen
from hybridsql.security import egress_gate

print(f"  {'verdict':<8}{'origin':<11}{'proven by':<30}text")
for segment in gen.build_segments(u, a):
    verdict = egress_gate.check_segment(segment, "notebook")
    print(f"  [{'pass' if verdict.allowed else 'BLOCK':<4}] {segment.origin:<11}"
          f"{verdict.verified_by:<30}{' '.join(segment.text.split())[:32]}...")

### The question: two layers, and what they are made of

**Layer 1, the exhaustive denylist.** We *own the database*, so we hold the complete
inventory of values for every indexed column. Matching against it is not an estimate,
it is exact — a value present in the database cannot slip past. This is the
difference from PII detection, which is a heuristic with a recall (MaskSQL measures
61.4 % on this task, so four values in ten leave in clear text).

**Layer 2, the allowlist, fail-closed.** For what the denylist cannot know: values of
non-indexed columns, misspellings, words nobody declared. A word that survives layer
1 is refused if it is a word of a short stored value.

Three sources, and only the first is written by hand. That proportion is the point:
the vocabulary a question may contain is mostly **read from the database itself**, so
it cannot drift away from it.

In [ ]:
import random

print(f"  {'text':<54}verdict")
for text in ["How many patients over :v2 received :v1?",
             "How many patients received aspirin?",
             "How many patients came from Skilled Nursing Facility?",
             "SELECT COUNT(*) FROM medication WHERE drugname = :v1"]:
    v = egress_gate.check(text, "notebook")
    print(f"  {text[:52]:<54}"
          f"{'passed' if v.allowed else 'REFUSED on ' + str(list(v.refused_tokens))}")

grammar = egress_gate.generic_vocabulary()
derived = egress_gate._schema_identifiers() | egress_gate.glossary_concepts()

print(f"  {'question grammar (written by hand)':<40}{len(grammar):>7}")
print(f"  {'schema identifiers (read from the db)':<40}{len(egress_gate._schema_identifiers()):>7}")
print(f"  {'glossary concepts (read from yaml)':<40}{len(egress_gate.glossary_concepts()):>7}")
print(f"  {'total allowed':<40}{len(egress_gate.allowlist()):>7}   "
      f"{100 * len(derived) / max(len(egress_gate.allowlist()), 1):.0f} % derived")
print(f"  {'words of stored values (blocked)':<40}{len(egress_gate.value_tokens()):>7}")
print(f"  {'complete values (blocked)':<40}{len(egress_gate.known_values()):>7}")

random.seed(7)
blocked = sorted(w for w in egress_gate.value_tokens() if w.isalpha() and len(w) > 4)
print("\n  a sample of the words a question may NOT contain")
for _ in range(3):
    print("    " + "  ".join(f"{w:<18}" for w in random.sample(blocked, 4)))

### Refusing too much is a bug as well

A gate has two ways of being wrong. It can let a value through — and it can refuse a
question that carries nothing to hide, which teaches the analyst to reword until
something passes. Only the first was ever measured, and the second was happening
constantly:

> *"How many laboratory records are **associated** with each patient ICU stay?"*
> → refused on `associated`.

`associated` is not data. In this database it appears only inside long hierarchical
diagnosis strings such as `hematology|coagulation disorders|DIC syndrome|associated
with intravascular clotting`. Every database's free text contains it.

Two changes, both rules rather than patches:

1. the blocked vocabulary now takes the words of **short values only** (three words
   or fewer) — a threshold already measured and used elsewhere in the same file, but
   never applied here. 8 708 words → 7 419. `aspirin`, `female`, `alive` and `hgb`
   stay; `associated`, `unique` and `necrotizing` go. Long values are still caught
   **whole** by layer 1;
2. the closed-class grammar was extended with the vocabulary of *asking a database a
   question* — `records`, `unique`, `names`, `one`. The same words serve a hospital,
   a bank or a warehouse; none can name a drug. A test fails the build if a word
   added there is ever also a whole stored value.

In [ ]:
print("  words that used to be refused in an ordinary question")
for word in ["associated", "records", "unique", "one", "administration"]:
    v = egress_gate.check(f"how many {word} per hospital", "notebook")
    print(f"    {word:<16}{'passes now' if v.allowed else 'still refused'}")

print("\n  and the values they might have hidden are still refused")
for value in ["Medical Records", "aspirin", "Female", "Skilled Nursing Facility"]:
    v = egress_gate.check(f"how many patients with {value}", "notebook")
    print(f"    {value:<26}{'refused, correct' if not v.allowed else 'PASSED — a leak'}")

### Both rates, measured

`scripts/measure_gate.py` replays every distinct value of the database through the
gate, and every question of the three evaluation sets through the whole pipeline.
Neither number means anything without the other.

In [ ]:
measurement = Path(os.environ["DB_PATH"]).with_name("gate_measurement.json")
if measurement.exists():
    g = json.loads(measurement.read_text())
    q = g.get("questions", {})
    print(f"  {'information-bearing values tested':<44}{g['values_bearing']:>8,}")
    print(f"  {'LEAK: how many cross the gate':<44}{g['passed_bearing']:>8,}"
          f"  {g['rate_bearing_pct']:>6.2f} %")
    print(f"  {'questions replayed end to end':<44}{q.get('questions_checked', 0):>8}")
    print(f"  {'REFUSAL: ordinary questions blocked':<44}{q.get('questions_refused', 0):>8}"
          f"  {q.get('refusal_rate_pct', 0):>6.2f} %")
else:
    print("  run scripts/measure_gate.py to produce this file")

The residual leak is explainable rather than mysterious: it is the words that are
**both a column name and a value**. `apacheapsvar` has columns `albumin`,
`creatinine`, `bun` and `wbc`, which are also lab names in `lab.labname`. Blocking
them would stop the model writing SQL against those columns.

### The proof: a real value, submitted on purpose

The gate is only worth something if it refuses.

In [ ]:
from hybridsql.security.egress_gate import LeakBlocked, Segment

try:
    egress_gate.require_segments(
        [Segment("How many patients received AMOXICILLIN 500 MG PO CAPS?", "question")],
        "check")
    print("  ALLOWED. This would be a failure.")
except LeakBlocked as blocked:
    print(f"  refused: {blocked}")

---

## 10. What is measured, and what would leave

Three annotated question sets, 133 questions. `complete` is the strict figure: every
value, every column and every name right in the same question — the only one that
describes what the user sees.

- **standard**, 79 questions naming something stored;
- **hard**, 28 written to break it — typos, slang, person names, values absent from
  this extract;
- **analytics**, 26 that name a **column** and aggregate over it, which is what an
  analyst actually types. Added the day the system answered *"the 10 most common
  diagnosis names"* with one row.

In [ ]:
results = PROJECT / "data/evaluation/understanding_results.json"
if results.exists():
    r = json.loads(results.read_text())
    print(f"  {'set':<12}{'questions':>10}{'extraction':>12}{'resolution':>12}{'complete':>10}")
    for name in ("standard", "hard", "analytics"):
        s = r.get(name)
        if s:
            print(f"  {name:<12}{s['questions']:>10}{s['extraction_recall_pct']:>11.1f}%"
                  f"{s['resolution_accuracy_pct']:>11.1f}%{s['full_understanding_pct']:>9.1f}%")
    o = r["overall"]
    print(f"\n  {o['correct_questions']}/{o['questions']} questions understood completely "
          f"({o['full_understanding_pct']} %)")

print(f"\n  the question    {u.question}")
print(f"  what is sent    {a.masked_question}")
print(f"  values out      0")
print(f"  kept here       {len(a.mapping)} value(s), {len(u.tables)} table name(s)")

---

Four architectures do different things with this. Three send it, one does not, and the difference is measurable.

**Next:** [3. Architectures](https://www.kaggle.com/code/kirazul/nl2sql-3-architectures)